# 03 — Gold AGREGADA para BI (Power BI · Kelly)

**Contrato §13.3 (datos del dashboard).** Materializa **tablas agregadas pequeñas** (no las 23M filas)
que responden la Pregunta de Oro vía las **dos palancas** del EDA. Todo sobre **base limpia**
(cuarentena 15–17 nov), salvo `agg_metricas_diarias` que conserva todos los días con banderas para
contar la calidad de datos.

> **Salida:** 6 CSV en un folder del Volume (`bi_export`). Descárgalos y **commitéalos en `reports/data/`**
> (excepción del `.gitignore`, §12.2); Power BI los consume desde ahí.
>
> **Consistencia:** se replican las definiciones del EDA (tabla UNIT, funnel, prize, segmentos) → los
> números coinciden con `eda_ecommerce.ipynb`. Si Kelly necesita un corte que no está, se añade una tabla más.

In [ ]:
from pyspark.sql import functions as F
import pandas as pd, os

SILVER = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
GOLD   = "/Volumes/workspace/default/e_commerce/gold/features_session"
BI_OUT = "/Volumes/workspace/default/e_commerce/gold/bi_export"   # descargar y commitear a reports/data/
os.makedirs(BI_OUT, exist_ok=True)

QUARANTINE = ["2019-11-15", "2019-11-16", "2019-11-17"]
silver_full = spark.read.format("delta").load(SILVER)
gold_full   = spark.read.format("delta").load(GOLD)
silver = silver_full.filter(~F.col("date").isin(QUARANTINE))     # base de NEGOCIO (limpia)
gold   = gold_full.filter(F.col("label_window_corrupt") == 0)
MISSING = "Unknown"

# Tabla UNIT (producto-en-sesion) sobre base limpia: identica al EDA -> numeros coinciden.
# En serverless .cache()/.persist() NO se soporta (PERSIST TABLE not supported) -> se materializa
# en un Delta temporal y se relee (mismo patron que el EDA). Se borra en la celda de cierre.
BI_UNITS = "/Volumes/workspace/default/e_commerce/gold/_tmp_bi_units"
(silver.groupBy("user_session", "product_id").agg(
    F.max(F.when(F.col("event_type") == "cart", 1).otherwise(0)).cast("boolean").alias("has_cart"),
    F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).cast("boolean").alias("has_purchase"),
    F.expr("percentile_approx(price, 0.5)").alias("price"),
    F.first("macro_category", ignorenulls=True).alias("macro_category"),
    F.first("brand", ignorenulls=True).alias("brand"))
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(BI_UNITS))
units = spark.read.format("delta").load(BI_UNITS)

# Crear las tablas csv's con Spark SQL puro
# Agrego Vistas Temporales (Temp Views) de los DataFrames base y luego ejecutar las consultas SQL.
# Registrar los DataFrames como vistas SQL temporales
silver_full.createOrReplaceTempView("vw_silver_full")
silver.createOrReplaceTempView("vw_silver")
units.createOrReplaceTempView("vw_units")


def to_csv(pdf, name):
    pdf.to_csv(f"{BI_OUT}/{name}.csv", index=True)
    print(f"  -> {name}.csv  ({pdf.shape[0]} filas x {pdf.shape[1]} cols)")
    return pdf

print("Bases limpias listas | UNIT:", units.count(), "| salida:", BI_OUT)

## 1. `agg_funnel_categoria` — ¿dónde se concentra la fuga? (Palanca A)
Gráfico sugerido: treemap / barras apiladas por categoría.

In [ ]:
#fc = (units.filter(F.col("macro_category").isNotNull() & (F.col("macro_category") != MISSING))
#      .groupBy("macro_category").agg(
#          F.count("*").alias("units"),
#          F.sum(F.when(F.col("has_cart") | F.col("has_purchase"), 1).otherwise(0)).alias("reached_cart"),
#          F.sum(F.when(F.col("has_purchase"), 1).otherwise(0)).alias("purchased"))
#      .toPandas().set_index("macro_category"))
#fc["cart_rate"]    = (fc["reached_cart"] / fc["units"] * 100).round(2)
#fc["conv_rate"]    = (fc["purchased"] / fc["units"] * 100).round(2)
#fc["cierre_pct"]   = (fc["purchased"] / fc["reached_cart"] * 100).round(2)
#fc["abandono_pct"] = (100 - fc["cierre_pct"]).round(2)
#fc = fc.sort_values("conv_rate", ascending=False)
#to_csv(fc, "agg_funnel_categoria")
#fc

df_agg_funnel_categoria = spark.sql("""
    SELECT 
        macro_category,
        COUNT(1) AS units,
        SUM(CAST(has_cart OR has_purchase AS INT)) AS reached_cart,
        SUM(CAST(has_purchase AS INT)) AS purchased,
        ROUND((SUM(CAST(has_cart OR has_purchase AS INT)) / COUNT(1)) * 100, 2) AS cart_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / COUNT(1)) * 100, 2) AS conv_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100, 2) AS cierre_pct,
        ROUND(100 - ((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100), 2) AS abandono_pct
    FROM vw_units
    WHERE macro_category IS NOT NULL AND macro_category != 'Unknown'
    GROUP BY macro_category
    ORDER BY conv_rate DESC
""")

fc = df_agg_funnel_categoria.toPandas().set_index("macro_category")
to_csv(fc, "agg_funnel_categoria")
display(fc)

## 2. `agg_revenue_en_juego` — ¿cuánto $ hay en carritos abandonados? (Palanca A)
Gráfico sugerido: treemap (área = revenue en juego).

In [ ]:
df_agg_revenue = spark.sql("""
    SELECT 
        macro_category,
        COUNT(1) AS carritos_abandonados,
        ROUND(SUM(price), 0) AS revenue_en_juego,
        ROUND(SUM(price) / COUNT(1), 2) AS ticket_medio
    FROM vw_units
    WHERE has_cart = TRUE AND has_purchase = FALSE
      AND macro_category IS NOT NULL AND macro_category != 'Unknown'
    GROUP BY macro_category
    ORDER BY revenue_en_juego DESC
""")

ab = df_agg_revenue.toPandas().set_index("macro_category")
to_csv(ab, "agg_revenue_en_juego")
display(ab)

#ab = (units.filter(F.col("has_cart") & (~F.col("has_purchase")) &
#                   F.col("macro_category").isNotNull() & (F.col("macro_category") != MISSING))
#      .groupBy("macro_category").agg(
#          F.count("*").alias("carritos_abandonados"),
#          F.round(F.sum("price"), 0).alias("revenue_en_juego"))
#      .toPandas().set_index("macro_category"))
#ab["ticket_medio"] = (ab["revenue_en_juego"] / ab["carritos_abandonados"]).round(2)
#ab = ab.sort_values("revenue_en_juego", ascending=False)
#to_csv(ab, "agg_revenue_en_juego")
#ab

## 3. `agg_marca_electronics` — ¿qué marcas concentran el premio? (Palanca A, drill)
Gráfico sugerido: barras (Samsung/Apple al frente).

In [ ]:
df_agg_marca = spark.sql("""
    SELECT 
        brand,
        COUNT(1) AS carritos,
        SUM(CAST(has_purchase AS INT)) AS comprados,
        ROUND(percentile_approx(price, 0.5), 2) AS ticket,
        COUNT(1) - SUM(CAST(has_purchase AS INT)) AS abandonados,
        ROUND(((COUNT(1) - SUM(CAST(has_purchase AS INT))) / COUNT(1)) * 100, 1) AS abandono_pct
    FROM vw_units
    WHERE macro_category = 'electronics'
      AND (has_cart = TRUE OR has_purchase = TRUE)
      AND brand IS NOT NULL AND brand != 'Unknown'
    GROUP BY brand
    HAVING COUNT(1) >= 100
    ORDER BY abandonados DESC
""")

me = df_agg_marca.toPandas().set_index("brand")
to_csv(me, "agg_marca_electronics")
display(me.head(15))


#elec = units.filter((F.col("macro_category") == "electronics") & (F.col("has_cart") | F.col("has_purchase")))
#me = (elec.filter(F.col("brand").isNotNull() & (F.col("brand") != MISSING))
#      .groupBy("brand").agg(
#          F.count("*").alias("carritos"),
#          F.sum(F.col("has_purchase").cast("int")).alias("comprados"),
#          F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("ticket"))
#      .toPandas().set_index("brand"))
#me["abandonados"]  = me["carritos"] - me["comprados"]
#me["abandono_pct"] = (me["abandonados"] / me["carritos"] * 100).round(1)
#me = me[me["carritos"] >= 100].sort_values("abandonados", ascending=False)
#to_csv(me, "agg_marca_electronics")
#me.head(15)

## 4. `agg_segmentos_comprador` — ¿qué segmento concentra el revenue? (Palanca B)
Gráfico sugerido: combo doble eje (% compradores vs % revenue). Ocasión = sesión distinta con compra.

In [ ]:
df_agg_segmentos = spark.sql("""
    WITH user_stats AS (
        SELECT 
            user_id,
            COUNT(DISTINCT user_session) AS ocasiones,
            SUM(price) AS revenue,
            CASE WHEN COUNT(DISTINCT user_session) >= 2 THEN 'recurrente' ELSE 'one-time' END AS segmento
        FROM vw_silver
        WHERE event_type = 'purchase'
        GROUP BY user_id
    ),
    segment_totals AS (
        SELECT 
            segmento,
            COUNT(1) AS n_compradores,
            ROUND(SUM(revenue), 0) AS revenue,
            ROUND(AVG(revenue), 2) AS ticket_promedio
        FROM user_stats
        GROUP BY segmento
    )
    SELECT 
        segmento,
        n_compradores,
        revenue,
        ticket_promedio,
        ROUND((n_compradores / SUM(n_compradores) OVER ()) * 100, 1) AS pct_compradores,
        ROUND((revenue / SUM(revenue) OVER ()) * 100, 1) AS pct_revenue
    FROM segment_totals
    ORDER BY segmento
""")

seg = df_agg_segmentos.toPandas().set_index("segmento")
to_csv(seg, "agg_segmentos_comprador")
display(seg)

#pur  = silver.filter(F.col("event_type") == "purchase")
#user = (pur.groupBy("user_id").agg(
#            F.countDistinct("user_session").alias("ocasiones"),
#            F.sum("price").alias("revenue"))
#        .withColumn("segmento", F.when(F.col("ocasiones") >= 2, "recurrente").otherwise("one-time")))
#seg = (user.groupBy("segmento").agg(
#          F.count("*").alias("n_compradores"),
#          F.round(F.sum("revenue"), 0).alias("revenue"),
#          F.round(F.avg("revenue"), 2).alias("ticket_promedio"))
#       .toPandas().set_index("segmento"))
#seg["pct_compradores"] = (seg["n_compradores"] / seg["n_compradores"].sum() * 100).round(1)
#seg["pct_revenue"]     = (seg["revenue"] / seg["revenue"].sum() * 100).round(1)
#seg = seg.sort_index()
#to_csv(seg, "agg_segmentos_comprador")
#seg

## 5. `agg_metricas_diarias` — evolución temporal (con banderas de calidad)
**Conserva TODOS los días** (incl. 15–17 nov) con `is_black_friday` y `ventana_corrupta` para que Kelly
cuente la historia de calidad de datos (anotación). Gráfico sugerido: líneas / áreas.

In [ ]:
df_agg_diarias = spark.sql("""
    SELECT 
        date,
        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS views,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS carts,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases,
        COALESCE(ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 0), 0) AS revenue,
        ROUND(SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) / 
              SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) * 100, 3) AS conv_x100,
        CASE WHEN CAST(date AS STRING) = '2019-11-29' THEN 1 ELSE 0 END AS is_black_friday,
        CASE WHEN CAST(date AS STRING) IN ('2019-11-15', '2019-11-16', '2019-11-17') THEN 1 ELSE 0 END AS ventana_corrupta
    FROM vw_silver_full
    GROUP BY date
    ORDER BY date
""")

daily = df_agg_diarias.toPandas().set_index("date")
to_csv(daily, "agg_metricas_diarias")
display(daily.tail(20))

#daily = (silver_full.groupBy("date").pivot("event_type", ["view", "cart", "purchase"]).count()
#         .toPandas().rename(columns={"view": "views", "cart": "carts", "purchase": "purchases"}).fillna(0))
#daily = daily.sort_values("date").set_index("date")
#rev = (silver_full.filter(F.col("event_type") == "purchase").groupBy("date")
#       .agg(F.round(F.sum("price"), 0).alias("revenue")).toPandas().set_index("date"))
#daily = daily.join(rev, how="left").fillna({"revenue": 0})
#daily["conv_x100"]       = (daily["purchases"] / daily["views"] * 100).round(3)
#_idx = daily.index.astype(str)
#daily["is_black_friday"] = (_idx == "2019-11-29").astype(int)
#daily["ventana_corrupta"] = _idx.isin(["2019-11-15", "2019-11-16", "2019-11-17"]).astype(int)
#to_csv(daily, "agg_metricas_diarias")
#daily.tail(20)

## 6. `agg_tipologia_visitante` — browser / intender / buyer (nivel sesión)
Gráfico sugerido: barras (reparto y conversión). buyer = compra; intender = carrito sin compra; browser = resto.

In [ ]:
df_agg_tipologia = spark.sql("""
    WITH session_types AS (
        SELECT 
            user_session,
            CASE 
                WHEN MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) = 1 THEN 'buyer'
                WHEN MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) = 1 THEN 'intender'
                ELSE 'browser'
            END AS tipo
        FROM vw_silver
        GROUP BY user_session
    )
    SELECT 
        tipo,
        COUNT(1) AS n_sesiones,
        ROUND((COUNT(1) / SUM(COUNT(1)) OVER ()) * 100, 2) AS pct
    FROM session_types
    GROUP BY tipo
""")

tip = df_agg_tipologia.toPandas().set_index("tipo").reindex(["browser", "intender", "buyer"])
to_csv(tip, "agg_tipologia_visitante")
display(tip)


#sess = (silver.groupBy("user_session").agg(
#            F.max(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("hc"),
#            F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("hp"))
#        .withColumn("tipo", F.when(F.col("hp") == 1, "buyer")
#                             .when(F.col("hc") == 1, "intender").otherwise("browser")))
#tip = sess.groupBy("tipo").agg(F.count("*").alias("n_sesiones")).toPandas().set_index("tipo")
#tip["pct"] = (tip["n_sesiones"] / tip["n_sesiones"].sum() * 100).round(2)
#tip = tip.reindex(["browser", "intender", "buyer"])
#to_csv(tip, "agg_tipologia_visitante")
#tip

In [ ]:
# 7. agg_funnel_global -- funnel GLOBAL incl. 'Unknown' (tarjeta KPI del titular).
#    AUDITORIA 5-jun: los CSV por-categoria (1 y 2) EXCLUYEN 'Unknown' (~32% de unidades) -> al sumarlos
#    NO se reproduce el titular (cart 3.93 / conv 2.24 / abandono 43.1, 994k carritos, $283.6M). Esta
#    tabla SI cuadra con el funnel global del EDA (eda_ecommerce.ipynb 4.1): 58.6M unidades incl. Unknown.



df_agg_global = spark.sql("""
    SELECT 
        'GLOBAL (incl. Unknown)' AS scope,
        COUNT(1) AS units,
        SUM(CAST(has_cart OR has_purchase AS INT)) AS reached_cart,
        SUM(CAST(has_purchase AS INT)) AS purchased,
        ROUND((SUM(CAST(has_cart OR has_purchase AS INT)) / COUNT(1)) * 100, 2) AS cart_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / COUNT(1)) * 100, 2) AS conv_rate,
        ROUND((SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT))) * 100, 2) AS cierre_pct,
        ROUND((1 - (SUM(CAST(has_purchase AS INT)) / SUM(CAST(has_cart OR has_purchase AS INT)))) * 100, 2) AS abandono_pct,
        SUM(CASE WHEN has_cart = TRUE AND has_purchase = FALSE THEN 1 ELSE 0 END) AS carritos_abandonados,
        ROUND(SUM(CASE WHEN has_cart = TRUE AND has_purchase = FALSE THEN price ELSE 0 END), 0) AS revenue_en_juego
    FROM vw_units
""")

fg = df_agg_global.toPandas().set_index("scope")
to_csv(fg, "agg_funnel_global")
display(fg)


#g = units.agg(
#    F.count("*").alias("units"),
#    F.sum(F.when(F.col("has_cart") | F.col("has_purchase"), 1).otherwise(0)).alias("reached_cart"),
#    F.sum(F.col("has_purchase").cast("int")).alias("purchased"),
#    F.sum(F.when(F.col("has_cart") & (~F.col("has_purchase")), 1).otherwise(0)).alias("carritos_abandonados"),
#    F.round(F.sum(F.when(F.col("has_cart") & (~F.col("has_purchase")), F.col("price")).otherwise(0.0)), 0).alias("revenue_en_juego"),
#).first()
#fg = pd.DataFrame([{
#    "scope": "GLOBAL (incl. Unknown)",
#    "units": g["units"], "reached_cart": g["reached_cart"], "purchased": g["purchased"],
#    "cart_rate": round(g["reached_cart"] / g["units"] * 100, 2),
#    "conv_rate": round(g["purchased"] / g["units"] * 100, 2),
#    "cierre_pct": round(g["purchased"] / g["reached_cart"] * 100, 2),
#    "abandono_pct": round((1 - g["purchased"] / g["reached_cart"]) * 100, 2),
#    "carritos_abandonados": g["carritos_abandonados"],
#    "revenue_en_juego": g["revenue_en_juego"],
#}]).set_index("scope")
#to_csv(fg, "agg_funnel_global")
#fg

## Cierre
Lista los CSV y recuerda el siguiente paso (commit a `reports/data/`).
El bloque del final borra el Delta temporal del EDA (`_tmp_eda_units`) — **descoméntalo cuando ya no lo uses** (Block 5).

In [ ]:
print("CSVs en", BI_OUT, ":")
for f in sorted(os.listdir(BI_OUT)):
    print("  -", f)
print("\nSiguiente: descargar estos 6 CSV y commitearlos en reports/data/ (los lee Power BI).")

# Limpieza del Delta temporal propio de este notebook (UNIT):
dbutils.fs.rm("/Volumes/workspace/default/e_commerce/gold/_tmp_bi_units", recurse=True)
print("borrado _tmp_bi_units")

# --- Block 5 (limpieza del Delta temporal del EDA) -- ejecutar cuando ya no se use ---
# dbutils.fs.rm("/Volumes/workspace/default/e_commerce/gold/_tmp_eda_units", recurse=True)